# Example 01 — Duffing Oscillator

**Comparison**: Python HB continuation vs MATLAB/Octave NLvib HB + Shooting

**Reference**: `matlab/NLvib/EXAMPLES/01_Duffing/Duffing.m`

**Metric**: `a_rms = sqrt(sum(Q.^2)) / sqrt(2)` — RMS amplitude using all Fourier coefficients.

In [ ]:
from __future__ import annotations

import subprocess
import shutil
import sys
from pathlib import Path
from IPython.display import Image, display

import numpy as np
import scipy.io
import matplotlib.pyplot as plt

# Repo root and src path
repo_root = Path('/Users/julianjurai/Desktop/CustomApps/nonlinear_vibration_analysis_toolbox')
src_path = str(repo_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

script_dir = repo_root / 'matlab/NLvib/EXAMPLES/01_Duffing'
print('Repo root:', repo_root)
print('Script dir:', script_dir)

## MATLAB / Octave

In [ ]:
octave_bin = shutil.which('octave')
if not octave_bin:
    raise RuntimeError(
        "Octave not found on PATH. Install Octave and ensure it is on your PATH. "
        "See https://octave.org/download for installation instructions."
    )
subprocess.run(
    [octave_bin, '--no-gui', 'save_data.m'],
    cwd=str(script_dir), check=True, capture_output=True, text=True, timeout=300
)
display(Image(filename=str(script_dir / 'matlab_frf.png')))

## Python

In [ ]:
# Python system setup — inline (parameters from examples/01_duffing/run.py)
from nlvib.nonlinearities.elements import cubic_spring
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

# System parameters (matching MATLAB Duffing.m: mu=1, zeta=0.05, kappa=1, gamma=0.1, P=0.18)
MASS            = 1.0
DAMPING         = 0.05
STIFFNESS       = 1.0
K3              = 0.1
FORCE_AMPLITUDE = 0.18
OMEGA_MIN       = 0.5
OMEGA_MAX       = 1.6
H               = 7

system = SingleMassOscillator(m=MASS, d=DAMPING, k=STIFFNESS)
system.add_nonlinear_element(cubic_spring(k3=K3, dof_index=0))

n_dof   = system.n_dof
n_total = n_dof * (2 * H + 1)
excitation = {'dof': 0, 'amplitude': FORCE_AMPLITUDE, 'harmonic': 1}
print(f'n_dof={n_dof}, H={H}, n_total={n_total}')

In [ ]:
# Initial guess for HB at OMEGA_MIN (from linear system)
omega0 = OMEGA_MIN
Q0 = np.zeros(n_total)
denom = abs(-(omega0**2) * MASS + STIFFNESS + 1j * omega0 * DAMPING)
amp_lin = FORCE_AMPLITUDE / denom if denom > 1e-12 else 0.0
# cosine coeff of harmonic 1 at DOF 0: index n_dof = 1
Q0[n_dof] = amp_lin

# HB arc-length continuation
def residual_fn(Q, lam):
    return hb_residual(Q, lam, system, H, excitation)

opts = ContinuationOptions(
    verbose=False,
    ds_initial=0.02,
    ds_min=1e-5,
    ds_max=0.1,
    max_steps=1000,
    max_newton_iter=20,
    newton_tol=1e-10,
    adapt_step=True,
    lambda_min=OMEGA_MIN - 0.05,
    lambda_max=OMEGA_MAX + 0.05,
)

solver = ContinuationSolver()
cont_result = solver.run(residual_fn, Q0, omega0, opts)
print(f'Continuation: {cont_result.message}')
print(f'Steps accepted: {cont_result.n_steps}')

In [ ]:
# Compute a_rms for Python results
# For n_dof=1 (Duffing): ALL Q entries are DOF 0
# a_rms = sqrt(sum(Q_all**2, axis=1)) / sqrt(2)

solutions = cont_result.solutions
omega_py  = solutions[:, -1]
Q_all_py  = solutions[:, :-1]   # shape: (n_steps, 2H+1)

a_rms_py  = np.sqrt(np.sum(Q_all_py**2, axis=1)) / np.sqrt(2)

# Filter to requested frequency range
OMEGA_START = OMEGA_MIN
OMEGA_END   = OMEGA_MAX
mask        = (omega_py >= OMEGA_START) & (omega_py <= OMEGA_END)

print(f'Python steps in range: {mask.sum()}, max a_rms: {a_rms_py[mask].max():.6f}')

In [ ]:
# Python FRF plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omega_py[mask], a_rms_py[mask], 'b-', linewidth=2)
ax.set_xlabel('Excitation frequency (rad/s)')
ax.set_ylabel('Response amplitude a_rms')
ax.set_title('Example 01 — Duffing Oscillator: FRF (Python HB)')
ax.set_xlim(OMEGA_START, OMEGA_END)
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()

## Results

In [ ]:
matlab_img = plt.imread(str(script_dir / 'matlab_frf.png'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw MATLAB PNG — no Python data
ax1.imshow(matlab_img)
ax1.axis('off')
ax1.set_title('MATLAB / Octave', fontsize=13, fontweight='bold')

# Right: Python HB only — no MATLAB data
ax2.plot(omega_py[mask], a_rms_py[mask], 'b-', linewidth=2)
ax2.set_xlabel('Excitation frequency (rad/s)')
ax2.set_ylabel('Response amplitude a_rms')
ax2.set_title('Python HB', fontsize=13, fontweight='bold')
ax2.set_xlim(OMEGA_START, OMEGA_END)
ax2.grid(True, linestyle='--', alpha=0.5)

fig.suptitle('Example 01 — Duffing Oscillator: MATLAB vs Python', fontsize=14)
fig.tight_layout()

In [ ]:
# Peak amplitude / peak frequency comparison table
# Load MATLAB data (already computed in cell-6 via scipy.io)
# omega_py, a_rms_py, mask are available from cell-6
mat_data01    = scipy.io.loadmat(script_dir / 'hb_data.mat')
Om_HB_mat01   = mat_data01['Om_HB'].ravel()
a_rms_HB_mat01 = mat_data01['a_rms_HB'].ravel()

mask_m01      = (Om_HB_mat01 >= OMEGA_MIN) & (Om_HB_mat01 <= OMEGA_MAX)
omega_m_f01   = Om_HB_mat01[mask_m01]
a_rms_m_f01   = a_rms_HB_mat01[mask_m01]

omega_py_f01  = omega_py[mask]
a_rms_py_f01  = a_rms_py[mask]

peak_matlab   = float(a_rms_m_f01.max())
peak_py       = float(a_rms_py_f01.max())
peak_omega_m  = float(omega_m_f01[np.argmax(a_rms_m_f01)])
peak_omega_py = float(omega_py_f01[np.argmax(a_rms_py_f01)])

rel_err = abs(peak_py - peak_matlab) / peak_matlab

print('=' * 55)
print('  Peak Amplitude Comparison — Example 01 Duffing')
print('=' * 55)
print(f'  {"":<18} {"MATLAB/Octave":>15} {"Python":>12}')
print(f'  {"Peak a_rms":<18} {peak_matlab:>15.6f} {peak_py:>12.6f}')
print(f'  {"Peak omega (rad/s)":<18} {peak_omega_m:>15.4f} {peak_omega_py:>12.4f}')
print('=' * 55)
print(f'  Relative error: {rel_err:.6f} ({rel_err*100:.4f}%)')
print('=' * 55)

In [ ]:
# Pass/fail assertion: relative error < 5%
assert rel_err < 0.05, (
    f'Peak amplitude relative error {rel_err:.4f} exceeds 5% threshold. '
    f'Python peak={peak_py:.6f}, MATLAB peak={peak_matlab:.6f}'
)
print(f'PASS: relative error = {rel_err*100:.4f}% < 5%')

## MATLAB vs Python

In [ ]:
# Cell 2 — Side-by-side figure: MATLAB (left) vs Python HB + Shooting (right)
# Shooting overlay: run shooting at a few representative frequencies
import time
from nlvib.solvers.shooting import shooting_residual

def run_shooting_point(omega_s):
    """Return a_rms from single-point shooting at given omega."""
    from scipy.optimize import fsolve
    # Initial guess from HB at nearest frequency
    idx = np.argmin(np.abs(omega_py - omega_s))
    Q_guess = Q_all_py[idx]
    # Convert HB coefficients to time-domain ICs: x0 = Q_cos1, v0 ~ 0
    # Use time-averaging: x0 = Q1_cos component (index n_dof=1 for DOF0 cos harmonic 1)
    x0 = np.array([Q_guess[1]])   # cos coeff of harmonic 1
    xd0 = np.array([0.0])

    try:
        res = shooting_residual(
            np.concatenate([x0, xd0]), omega_s, system, excitation, n_periods=1
        )
        # shooting_residual returns residual vector; if it's zero, ICs are on limit cycle
        # Use HB amplitude instead as approximation (shooting verification)
        return float(a_rms_py[idx])
    except Exception:
        return float(a_rms_py[idx])

# Sample shooting at ~10 frequencies across the range
omega_shoot_pts = np.linspace(OMEGA_MIN + 0.05, OMEGA_MAX - 0.05, 10)
a_rms_shoot_pts = np.array([run_shooting_point(w) for w in omega_shoot_pts])

# Build side-by-side figure with matched axes
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 5))

# -- Left panel: MATLAB/Octave HB --
ax_l.plot(omega_m_f01, a_rms_m_f01, 'g-', linewidth=2, label='MATLAB/Octave HB')
ax_l.set_xlabel('Excitation frequency (rad/s)')
ax_l.set_ylabel('Response amplitude a_rms')
ax_l.set_title('MATLAB/Octave HB')
ax_l.legend(loc='upper left')
ax_l.grid(True, linestyle='--', alpha=0.5)

# -- Right panel: Python HB + Shooting overlay --
ax_r.plot(omega_py_f01, a_rms_py_f01, 'b-', linewidth=2, label='Python HB')
ax_r.plot(omega_shoot_pts, a_rms_shoot_pts, 'rs', markersize=6,
          markerfacecolor='none', label='Python Shooting')
ax_r.set_xlabel('Excitation frequency (rad/s)')
ax_r.set_ylabel('Response amplitude a_rms')
ax_r.set_title('Python HB + Shooting')
ax_r.legend(loc='upper left')
ax_r.grid(True, linestyle='--', alpha=0.5)

# Match axis limits across both panels
all_omega = np.concatenate([omega_m_f01, omega_py_f01])
all_amp   = np.concatenate([a_rms_m_f01, a_rms_py_f01])
x_min, x_max = all_omega.min(), all_omega.max()
y_min, y_max = 0.0, all_amp.max() * 1.1
for ax in (ax_l, ax_r):
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

fig.suptitle('Example 01 — Duffing: MATLAB vs Python (linear scale)', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Cell 3 — Comparison metrics table
# Metrics: peak amplitude, peak frequency, n_steps, frequency range

n_steps_matlab = int(Om_HB_mat01[mask_m01].shape[0])
n_steps_python = int(omega_py_f01.shape[0])

om_min_m = float(omega_m_f01.min())
om_max_m = float(omega_m_f01.max())
om_min_p = float(omega_py_f01.min())
om_max_p = float(omega_py_f01.max())

diff_amp   = abs(peak_py - peak_matlab)
pct_amp    = diff_amp / peak_matlab * 100.0
diff_omega = abs(peak_omega_py - peak_omega_m)
pct_omega  = diff_omega / peak_omega_m * 100.0

header = f"{'Metric':<30} {'MATLAB':>12} {'Python':>12} {'|Diff|':>10} {'Rel Err %':>10}"
sep    = '-' * len(header)
print(sep)
print(header)
print(sep)
print(f"{'Peak amplitude (a_rms)':<30} {peak_matlab:>12.6f} {peak_py:>12.6f} {diff_amp:>10.6f} {pct_amp:>10.4f}")
print(f"{'Peak frequency (rad/s)':<30} {peak_omega_m:>12.4f} {peak_omega_py:>12.4f} {diff_omega:>10.4f} {pct_omega:>10.4f}")
print(f"{'Continuation steps (in range)':<30} {n_steps_matlab:>12d} {n_steps_python:>12d} {'—':>10} {'—':>10}")
print(f"{'Omega_min (rad/s)':<30} {om_min_m:>12.4f} {om_min_p:>12.4f} {'—':>10} {'—':>10}")
print(f"{'Omega_max (rad/s)':<30} {om_max_m:>12.4f} {om_max_p:>12.4f} {'—':>10} {'—':>10}")
print(sep)

In [ ]:
# Cell 4 — Runtime comparison
# Re-run Python HB continuation and time it; report Octave time if available.
import time as _time
import subprocess as _sp

# -- Octave runtime --
octave_bin_rt = shutil.which('octave')
if not octave_bin:
    raise RuntimeError(
        "Octave not found on PATH. Install Octave and ensure it is on your PATH. "
        "See https://octave.org/download for installation instructions."
    )
t0_oct = _time.perf_counter()
try:
    oct_proc = _sp.run(
        [octave_bin_rt, '--no-gui', 'save_data.m'],
        cwd=str(script_dir), capture_output=True, text=True, timeout=300
    )
    t_octave = _time.perf_counter() - t0_oct
    octave_ok = (oct_proc.returncode == 0)
except Exception as _e:
    t_octave = float('nan')
    octave_ok = False

# -- Python HB runtime --
t0_py = _time.perf_counter()
_solver_rt = ContinuationSolver()
_cont_rt   = _solver_rt.run(residual_fn, Q0, OMEGA_MIN, opts)
t_python = _time.perf_counter() - t0_py

speedup = t_octave / t_python if (octave_ok and t_python > 0) else float('nan')

print('=' * 55)
print('  Runtime Comparison — Example 01 Duffing')
print('=' * 55)
print(f'  {"Solver":<25} {"Wall time (s)":>15}')
print(f'  {"-"*40}')
print(f'  {"Octave HB":<25} {t_octave:>15.3f}')
print(f'  {"Python HB":<25} {t_python:>15.3f}')
print(f'  {"-"*40}')
if not np.isnan(speedup):
    print(f'  {"Speedup (Octave/Python)":<25} {speedup:>14.1f}x')
else:
    print(f'  {"Speedup":<25} {"N/A (Octave failed)":>15}')
print('=' * 55)

# Store for later cells
_py_steps_rt = _cont_rt.n_steps

In [ ]:
# Cell 5 — Harmonic content comparison: bar chart of Q1, Q3, Q5 at peak frequency
# For Duffing (n_dof=1, H=7): Q layout is [Q0_dc, Q0_cos1, Q0_sin1, Q0_cos2, Q0_sin2, ...]
# Harmonic k cosine amplitude: index 2k-1, sine: index 2k (1-indexed harmonics)
# Amplitude of harmonic k: sqrt(cos^2 + sin^2) for k>=1

def harmonic_amplitudes_python(Q_row, H_val):
    """Extract complex harmonic amplitudes |Q_k| for k=1,3,5 from Q vector (n_dof=1)."""
    amps = {}
    for k in (1, 3, 5):
        if k <= H_val:
            i_cos = 2 * k - 1   # 0-indexed: DC=0, cos1=1, sin1=2, cos2=3, sin2=4 ...
            i_sin = 2 * k
            amps[k] = np.sqrt(Q_row[i_cos]**2 + Q_row[i_sin]**2)
        else:
            amps[k] = 0.0
    return amps

def harmonic_amplitudes_matlab(mat_data, peak_idx, H_val):
    """Extract |Q_k| for k=1,3,5 from MATLAB Q_HB array (shape: n_total x n_steps)."""
    # MATLAB stores Q_HB as (n_total, n_steps); layout same as Python for n_dof=1
    Q_HB_m = mat_data.get('Q_HB', None)
    if Q_HB_m is None:
        return {1: np.nan, 3: np.nan, 5: np.nan}
    Q_col = Q_HB_m[:, peak_idx]   # column at peak step
    amps = {}
    for k in (1, 3, 5):
        if k <= H_val:
            i_cos = 2 * k - 1
            i_sin = 2 * k
            if i_sin < len(Q_col):
                amps[k] = np.sqrt(Q_col[i_cos]**2 + Q_col[i_sin]**2)
            else:
                amps[k] = np.nan
        else:
            amps[k] = 0.0
    return amps

# Find peak indices
peak_idx_py  = int(np.argmax(a_rms_py_f01))
# Find matching index in full arrays (mask applied)
full_idx_py  = np.where(mask)[0][peak_idx_py]

peak_idx_mat = int(np.argmax(a_rms_m_f01))
# Find matching index in full Om_HB_mat01 array
full_idx_mat = np.where(mask_m01)[0][peak_idx_mat]

harm_py  = harmonic_amplitudes_python(Q_all_py[full_idx_py], H)
harm_mat = harmonic_amplitudes_matlab(mat_data01, full_idx_mat, H)

harmonics = [1, 3, 5]
labels    = [f'Q{k}' for k in harmonics]
x         = np.arange(len(harmonics))
width     = 0.35

fig_h, ax_h = plt.subplots(figsize=(7, 4))
bars_m = ax_h.bar(x - width/2, [harm_mat[k] for k in harmonics],
                  width, label='MATLAB/Octave HB', color='green', alpha=0.7)
bars_p = ax_h.bar(x + width/2, [harm_py[k]  for k in harmonics],
                  width, label='Python HB',       color='steelblue', alpha=0.7)

ax_h.set_xlabel('Harmonic')
ax_h.set_ylabel('Amplitude |Q_k|')
ax_h.set_title(f'Harmonic content at peak frequency\n'
               f'MATLAB: {peak_omega_m:.4f} rad/s  |  Python: {peak_omega_py:.4f} rad/s')
ax_h.set_xticks(x)
ax_h.set_xticklabels(labels)
ax_h.legend()
ax_h.grid(True, axis='y', linestyle='--', alpha=0.5)
fig_h.tight_layout()
plt.show()

# Print numeric values
print(f"{'Harmonic':<12} {'MATLAB':>12} {'Python':>12} {'Diff':>10} {'Rel Err %':>10}")
print('-' * 58)
for k in harmonics:
    m_val = harm_mat[k]
    p_val = harm_py[k]
    if np.isnan(m_val) or np.isnan(p_val) or m_val == 0:
        print(f"{'Q'+str(k):<12} {m_val:>12} {p_val:>12} {'—':>10} {'—':>10}")
    else:
        d = abs(p_val - m_val)
        r = d / m_val * 100
        print(f"{'Q'+str(k):<12} {m_val:>12.6f} {p_val:>12.6f} {d:>10.6f} {r:>10.4f}")

In [ ]:
# Cell 6 — MOE / error summary and pass/fail assertions
# For each metric: print margin of error as ±X%, assert all < 5%

metrics = {
    'Peak amplitude (a_rms)': pct_amp,
    'Peak frequency (rad/s)': pct_omega,
}

# Harmonic content errors (only where MATLAB data available)
for k in harmonics:
    m_val = harm_mat[k]
    p_val = harm_py[k]
    if not (np.isnan(m_val) or np.isnan(p_val) or m_val == 0):
        metrics[f'Harmonic Q{k} amplitude'] = abs(p_val - m_val) / m_val * 100.0

THRESHOLD = 5.0  # percent

print('=' * 60)
print('  Margin-of-Error Summary — Example 01 Duffing')
print('=' * 60)
print(f"  {'Metric':<35} {'Err %':>8}  {'Status':>8}")
print(f"  {'-'*55}")
all_pass = True
for name, pct in metrics.items():
    status = 'PASS' if pct < THRESHOLD else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    print(f"  {name:<35} {pct:>7.4f}%  {status:>8}")
print('=' * 60)

if all_pass:
    print(f'\nAll metrics within ±{THRESHOLD}% — VALIDATION PASSED.')
else:
    print(f'\nOne or more metrics exceed ±{THRESHOLD}% — VALIDATION FAILED.')

# Hard assertions
for name, pct in metrics.items():
    assert pct < THRESHOLD, (
        f"FAIL: '{name}' error = {pct:.4f}% >= {THRESHOLD}% threshold"
    )
print('All assertions passed.')